# Track 2 · Stage 1 — Generate the code-mixed **text**

Replication of Biswas et al., Interspeech 2025 (Track 2).

Few-shot-prompt **Gemma 4 E4B** for Hindi-English bigrams → filter them → expand each
into four sentences (~16k). Push the text to `RohanRamesh/hi-en-synth-cs`.

Audio synthesis is a **separate notebook** (`01b`). `parler-tts` hard-pins
`transformers==4.46.1`, Gemma 4 needs `transformers>=5.5` — they cannot coexist in one
process. The Hub is the checkpoint between them.

### The model: `google/gemma-4-E4B-it`
* **apache-2.0, ungated** — no licence click-through, no gated-repo token scope, no mirror
  needed. (Gemma 3's `google/*` repos *are* gated; Gemma 4's are not.)
* Deviation **D1**: the paper used Llama-3.3-70B (141 GB bf16). Gemma 4 E4B is ~8B raw
  params, pretrained on 140+ languages, and a **deterministic script filter** sits behind it
  to catch what it still gets wrong.
* The one **live risk**: Gemma 4 is **bf16-native** (`torch_dtype: bfloat16`) and the T4 is
  Turing — it has **no bf16**. fp16 activations can exceed 65,504 and go non-finite.
  `backend.py` probes the logits after loading and transparently reloads with float32 compute
  if they are NaN/inf. **The smoke test below is how you find out**, in about a minute.
* Two non-issues, confirmed against the real config (the backend defends against both anyway):
  it is `Gemma4ForConditionalGeneration` but *is* registered under `AutoModelForCausalLM`, and
  its chat template **does** support a `system` role — so the paper's verbatim system prompt
  survives intact.

### Before you run
* HF **write** token in Kaggle Secrets as `HF_TOKEN`. Internet **on**, **GPU T4 ×2**.
* Nothing to accept — Gemma 4 is apache-2.0. (Parler-TTS *is* gated; that bites in `01b`.)

Runtime ≈ 2h. Every LLM call is cached, so a 12h timeout costs nothing on a re-run.

## 0 · Install

In [1]:
# Gemma 4 needs transformers >= 5.5. NO parler-tts here -- that is 01b's problem.
!pip install -q -U "transformers>=5.5" accelerate bitsandbytes
!pip install -q "datasets<4" librosa soundfile soxr omegaconf rich
!pip install -q --force-reinstall --no-deps git+https://github.com/BRUH-MAIN/codeswitching.git

# This NOTEBOOK's own version. `pip install` updates the csasr PACKAGE but NOT the
# .ipynb -- an old notebook against a new package is a real and confusing failure
# (the old 01a loaded the model in-kernel and starved every subprocess of VRAM).
NOTEBOOK_VERSION = "0.9.0"

import csasr, transformers
assert csasr.__version__ == NOTEBOOK_VERSION, (
    f"csasr package is {csasr.__version__} but this NOTEBOOK is {NOTEBOOK_VERSION}.\n"
    "  package older  -> restart the kernel (Run > Restart & clear); pip skips a\n"
    "                    reinstall when the version looks satisfied.\n"
    "  notebook older -> re-download it from the repo; pip does NOT update .ipynb files."
)
from packaging.version import Version
assert Version(transformers.__version__) >= Version("5.5"), (
    f"transformers {transformers.__version__} cannot load Gemma 4 (needs >= 5.5)."
)
print("csasr", csasr.__version__, "| transformers", transformers.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 101.5 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.7 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
csasr 0.9.0 | transformers 5.13.1


In [2]:
import os, subprocess, sys
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
HF_TOKEN = os.environ["HF_TOKEN"]

REAL_REPO  = "RohanRamesh/mucs-he-cs"
SYNTH_REPO = "RohanRamesh/hi-en-synth-cs"
LLM        = "google/gemma-4-E4B-it"      # apache-2.0, ungated. E2B is the smaller option.

WORK  = Path("/kaggle/working")
MAN   = WORK / "manifests"; MAN.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "llm_cache"; CACHE.mkdir(parents=True, exist_ok=True)

def run(*args):
    print(">", " ".join(str(a) for a in args), flush=True)
    p = subprocess.run([sys.executable, "-m", *args])   # inherits HF_TOKEN + streams
    if p.returncode != 0:
        raise RuntimeError(
            f"{args[0]} failed (exit {p.returncode}). The real error is printed ABOVE "
            f"this traceback - scroll up in this cell's output."
        )

import torch
N_GPU = max(1, torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}  {free/2**30:5.1f} GiB free / {total/2**30:.1f} GiB")
    assert free / total > 0.85, (
        f"cuda:{i} already has {(total-free)/2**30:.1f} GiB in use. Something in THIS "
        "kernel is holding the GPU (an older notebook loaded the model in-cell). "
        "Every subprocess below will then fail with 'Some modules are dispatched on "
        "the CPU or the disk'. Restart the kernel: Run > Restart & clear all."
    )

from csasr.manifest import read_jsonl, write_jsonl

def run_sharded(module, out_path, *args, cache_stem="cache"):
    """Run one process PER GPU and merge the shards.

    `device_map="auto"` puts the whole model on ONE card, so a single process
    leaves Kaggle's second T4 completely idle. Sharding the work across two
    processes -- each pinned to its own GPU with CUDA_VISIBLE_DEVICES -- is a
    straight 2x. Each shard gets its OWN cache file: two writers on one JSONL
    would interleave and corrupt it.
    """
    out_path = Path(out_path)
    procs = []
    for i in range(N_GPU):
        cmd = [sys.executable, "-m", module, *[str(a) for a in args],
               "--cache", str(CACHE / f"{cache_stem}.shard{i}.jsonl"),
               "--out", str(out_path.with_suffix(f".shard{i}.jsonl")),
               "--shard", str(i), "--num-shards", str(N_GPU)]
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        print(f"> GPU{i}: {module} shard {i}/{N_GPU}", flush=True)
        procs.append(subprocess.Popen(cmd, env=env))

    for i, p in enumerate(procs):
        if p.wait() != 0:
            raise RuntimeError(f"{module} shard {i} failed (exit {p.returncode}). "
                               "The real error is printed ABOVE - scroll up.")

    rows = [r for i in range(N_GPU)
            for r in read_jsonl(out_path.with_suffix(f".shard{i}.jsonl"))]
    write_jsonl(out_path, rows)
    print(f"merged {len(rows):,} rows -> {out_path}")
    return rows

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


cuda:0   14.5 GiB free / 14.6 GiB
cuda:1   14.5 GiB free / 14.6 GiB


## 1 · Pull the in-domain transcripts (few-shot exemplars)

Track 2 never trains on real code-switched audio — we only need the *text* of the MUCS train split.

In [3]:
from datasets import load_dataset
from csasr.manifest import write_jsonl

train_text = load_dataset(REAL_REPO, "train_text", split="train", token=HF_TOKEN)
write_jsonl(MAN / "mucs_train.jsonl", [dict(r) for r in train_text])
print(f"{len(train_text):,} in-domain sentences for few-shot prompting")
print(train_text[0]["text"])

README.md: 0.00B [00:00, ?B/s]

train_text/train-00000-of-00001.parquet:   0%|          | 0.00/2.96M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52825 [00:00<?, ? examples/s]

52,825 in-domain sentences for few-shot prompting
दोस्तों bash में nested और multilevel if statement के spoken tutorial में आपका स्वागत है


## 2 · SMOKE TEST — 20 bigrams before committing 2 hours

Loads Gemma, runs the **fp16 → float32 logits health check** (Gemma 4 is bf16-native and
the T4 has no bf16), generates real bigrams, and shows which survive the script filter.

**It runs as a subprocess, deliberately.** Jupyter's `Out[]` history holds a reference to
anything a cell produced, so `del model` does *not* free the VRAM — and the next subprocess
then dies with `Some modules are dispatched on the CPU or the disk`. Keeping the model out
of the kernel entirely is the only reliable fix.

Expect Gemma to emit **three**-word phrases (`बुनियादी formatting basics`). That is fine: the
filter extracts the switch pair from inside them. See deviation **D9**.

In [4]:
run("csasr.llm.smoke", "--model", LLM,
    "--train-manifest", MAN / "mucs_train.jsonl",
    "--n-calls", "2", "--bigrams-per-call", "10")

> csasr.llm.smoke --model google/gemma-4-E4B-it --train-manifest /kaggle/working/manifests/mucs_train.jsonl --n-calls 2 --bigrams-per-call 10


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


[smoke] model  : google/gemma-4-E4B-it
[smoke] prompts: 2 x 10 bigrams



smoke (cached 0): 100%|██████████| 2/2 [00:07<00:00,  3.58s/req]


## 3 · Generate bigrams

Paper: 44,657 raw → 5,932 unique (13.3%).

In [5]:
run_sharded("csasr.llm.gen_bigrams", MAN / "bigrams_raw.jsonl",
            "--train-manifest", MAN / "mucs_train.jsonl",
            "--model", LLM, "--n-calls", "4466", "--batch-size", "32",
            cache_stem="bigrams")

> GPU0: csasr.llm.gen_bigrams shard 0/2
> GPU1: csasr.llm.gen_bigrams shard 1/2


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
bigrams (cached 0): 100%|██████████| 2233/2233 [23:36<00:00,  1.58req/s]


merged 45,938 rows -> /kaggle/working/manifests/bigrams_raw.jsonl


[{'bigram': 'दाईं ओर screen', 'call_id': 0},
 {'bigram': 'इस tutorial', 'call_id': 0},
 {'bigram': 'हमने निम्न document', 'call_id': 0},
 {'bigram': 'करना सीखा English', 'call_id': 0},
 {'bigram': 'फिर type ls', 'call_id': 0},
 {'bigram': 'enter दबाएं terminal', 'call_id': 0},
 {'bigram': 'स्क्रीन पर window', 'call_id': 0},
 {'bigram': 'टर्मिनल विंडो', 'call_id': 0},
 {'bigram': 'hello world type', 'call_id': 0},
 {'bigram': 'बुनियादी concepts', 'call_id': 2},
 {'bigram': 'परियोजना management', 'call_id': 2},
 {'bigram': 'डेटा forms', 'call_id': 2},
 {'bigram': 'सॉफ्टवेयर application', 'call_id': 2},
 {'bigram': 'प्रस्तुति presentation', 'call_id': 2},
 {'bigram': 'शिक्षकों teacher', 'call_id': 2},
 {'bigram': 'भारत सरकार government', 'call_id': 2},
 {'bigram': 'परिचय introduction', 'call_id': 2},
 {'bigram': 'तरीक़ा method', 'call_id': 2},
 {'bigram': 'समाप्ति completion', 'call_id': 2},
 {'bigram': 'कार्यशालाओं workshop', 'call_id': 4},
 {'bigram': 'प्रमाणपत्र certificate', 'call_id'

## 4 · Filter

Deterministic script filter (one Devanagari token + one Latin token), then an LLM translation check with 3-sample self-consistency.
Paper: 5,932 unique → 5,477 valid (92.3%).

In [6]:
run_sharded("csasr.llm.filter_bigrams", MAN / "bigrams_valid.jsonl",
            "--raw", MAN / "bigrams_raw.jsonl",
            "--model", LLM, "--items-per-call", "20", "--n-samples", "3",
            "--batch-size", "16",
            cache_stem="transcheck")

> GPU0: csasr.llm.filter_bigrams shard 0/2
> GPU1: csasr.llm.filter_bigrams shard 1/2


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
transcheck 3/3 (cached 0): 100%|██████████| 369/369 [07:48<00:00,  1.27s/req]


merged 10,078 rows -> /kaggle/working/manifests/bigrams_valid.jsonl


[{'bigram': '#file मेन्यु',
  'hi_word': 'मेन्यु',
  'en_word': '#file',
  'checks': {'script': True, 'not_translation': True}},
 {'bigram': '#कैसे tutorial',
  'hi_word': '#कैसे',
  'en_word': 'tutorial',
  'checks': {'script': True, 'not_translation': True}},
 {'bigram': '#यह process',
  'hi_word': '#यह',
  'en_word': 'process',
  'checks': {'script': True, 'not_translation': True}},
 {'bigram': 'करें enter',
  'hi_word': 'करें',
  'en_word': 'enter',
  'checks': {'script': True, 'not_translation': True}},
 {'bigram': '2methyl13butadiene दिखती',
  'hi_word': 'दिखती',
  'en_word': '2methyl13butadiene',
  'checks': {'script': True, 'not_translation': True}},
 {'bigram': 'tool प्रयोग',
  'hi_word': 'प्रयोग',
  'en_word': 'tool',
  'checks': {'script': True, 'not_translation': True}},
 {'bigram': 'IIT बॉम्बे',
  'hi_word': 'बॉम्बे',
  'en_word': 'IIT',
  'checks': {'script': True, 'not_translation': True}},
 {'bigram': 'Java में',
  'hi_word': 'में',
  'en_word': 'Java',
  'checks': {'sc

## 5 · Expand each bigram into four sentences

2 English-matrix, 2 Hindi-matrix. Paper: ~16,000 unique from a theoretical 21,908.

In [7]:
run_sharded("csasr.llm.gen_sentences", MAN / "sentences.jsonl",
            "--bigrams", MAN / "bigrams_valid.jsonl",
            "--model", LLM, "--batch-size", "32",
            cache_stem="sentences")

> GPU0: csasr.llm.gen_sentences shard 0/2
> GPU1: csasr.llm.gen_sentences shard 1/2


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
sentences (cached 0): 100%|██████████| 5039/5039 [42:47<00:00,  1.96req/s]


merged 15,733 rows -> /kaggle/working/manifests/sentences.jsonl


[{'sent_id': '0809deadf0ae84c1',
  'text': 'मुझे प्रयोगशाला में 2methyl13butadiene दिखती हुई मिली।',
  'bigram': '2methyl13butadiene दिखती',
  'matrix_lang': 'hi'},
 {'sent_id': '460473da888c643b',
  'text': 'He dreams of getting into IIT बॉम्बे.',
  'bigram': 'IIT बॉम्बे',
  'matrix_lang': 'en'},
 {'sent_id': '564ba1148ec5093d',
  'text': 'Studying at IIT बॉम्बे is a huge achievement.',
  'bigram': 'IIT बॉम्बे',
  'matrix_lang': 'en'},
 {'sent_id': '6e0c2d4e1fce473b',
  'text': 'IIT बॉम्बे में प्रवेश पाना हर छात्र का सपना होता है।',
  'bigram': 'IIT बॉम्बे',
  'matrix_lang': 'hi'},
 {'sent_id': 'c0da71b37aacdde9',
  'text': 'IIT बॉम्बे की प्रतिष्ठा बहुत ऊँची है।',
  'bigram': 'IIT बॉम्बे',
  'matrix_lang': 'hi'},
 {'sent_id': '397ab5eefb0556e7',
  'text': 'We need to understand the operation of MOSFETs उसी तरह जैसे we discussed earlier.',
  'bigram': 'MOSFETs उसी',
  'matrix_lang': 'en'},
 {'sent_id': 'd69a64d48f52e86e',
  'text': 'The comparison between MOSFETs उसी और other semicondu

### GATE 1 — yields must track the paper

A large divergence in the 13.3% dedup rate means the prompt or temperature is off. Decide here, not after 6 hours of TTS.

In [8]:
raw   = list(read_jsonl(MAN / "bigrams_raw.jsonl"))
uniq  = {r["bigram"] for r in raw}
valid = list(read_jsonl(MAN / "bigrams_valid.jsonl"))
sents = list(read_jsonl(MAN / "sentences.jsonl"))

rows = [
    ("raw bigrams",    len(raw),   44_657, None),
    ("unique bigrams", len(uniq),   5_932, len(uniq) / max(len(raw), 1)),
    ("valid bigrams",  len(valid),  5_477, len(valid) / max(len(uniq), 1)),
    ("sentences",      len(sents), 16_000, None),
]
print(f"{'metric':<16}{'ours':>10}{'paper':>10}{'survival':>12}")
for name, got, want, surv in rows:
    s = f"{surv:.1%}" if surv else "-"
    print(f"{name:<16}{got:>10,}{want:>10,}{s:>12}")
print("\npaper survival: dedup 13.3%, filter 92.3%")
print("\nGemma 4 E4B is far smaller than the paper's 70B (deviation D1), so a lower")
print("valid-bigram yield is expected. What matters is that ENOUGH sentences survive:")
print(f"  -> {len(sents):,} sentences  (need >~8,000 for a usable 22h corpus)")

for r in sents[:5]:
    print("   ", r["text"])

metric                ours     paper    survival
raw bigrams         45,938    44,657           -
unique bigrams      26,721     5,932       58.2%
valid bigrams       10,078     5,477       37.7%
sentences           15,733    16,000           -

paper survival: dedup 13.3%, filter 92.3%

Gemma 4 E4B is far smaller than the paper's 70B (deviation D1), so a lower
valid-bigram yield is expected. What matters is that ENOUGH sentences survive:
  -> 15,733 sentences  (need >~8,000 for a usable 22h corpus)
    मुझे प्रयोगशाला में 2methyl13butadiene दिखती हुई मिली।
    He dreams of getting into IIT बॉम्बे.
    Studying at IIT बॉम्बे is a huge achievement.
    IIT बॉम्बे में प्रवेश पाना हर छात्र का सपना होता है।
    IIT बॉम्बे की प्रतिष्ठा बहुत ऊँची है।


### Repair — strip the matrix-language labels

Gemma prefixes each sentence with its matrix language:

    English: Many software programs have different aliases निर्धारित for commands.

Left in, Parler-TTS would literally **speak** "English colon, many software programs…" and
Whisper would then be **trained to emit `English:`** at the start of every transcript. This
re-cleans the text, re-checks that the bigram survived, and recomputes the matrix language
(the prefix biased it). Seconds — no LLM re-run.

In [9]:
run("csasr.llm.fix_sentences",
    "--in", MAN / "sentences.jsonl",
    "--bigrams", MAN / "bigrams_valid.jsonl",
    "--out", MAN / "sentences.jsonl")

sents = list(read_jsonl(MAN / "sentences.jsonl"))
print(f"\n{len(sents):,} sentences ready for TTS:")
for r in sents[:5]:
    print("   ", r["text"])

> csasr.llm.fix_sentences --in /kaggle/working/manifests/sentences.jsonl --bigrams /kaggle/working/manifests/bigrams_valid.jsonl --out /kaggle/working/manifests/sentences.jsonl

15,733 sentences ready for TTS:
    मुझे प्रयोगशाला में 2methyl13butadiene दिखती हुई मिली।
    He dreams of getting into IIT बॉम्बे.
    Studying at IIT बॉम्बे is a huge achievement.
    IIT बॉम्बे में प्रवेश पाना हर छात्र का सपना होता है।
    IIT बॉम्बे की प्रतिष्ठा बहुत ऊँची है।


## 6 · Push the text to the Hub

This is the handoff to `01b`. Push before anything can time out.

In [10]:
for man, cfg in [("bigrams_valid.jsonl", "bigrams"), ("sentences.jsonl", "sentences")]:
    run("csasr.data.push_to_hub", "--manifest", MAN / man,
        "--repo", SYNTH_REPO, "--config", cfg, "--text-only")
print("\ntext stage complete -> now run 01b_synthesize_audio.ipynb")

> csasr.data.push_to_hub --manifest /kaggle/working/manifests/bigrams_valid.jsonl --repo RohanRamesh/hi-en-synth-cs --config bigrams --text-only


Creating parquet from Arrow format: 100%|██████████| 11/11 [00:00<00:00, 614.44ba/s]
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|██████████|  260kB /  260kB            

Processing Files (1 / 1)      : 100%|██████████|  260kB /  260kB,  650kB/s  
New Data Upload               : 100%|██████████|  260kB /  260kB,  650kB/s  

                              : 100%|██████████|  260kB /  260kB            

Processing Files (1 / 1)      : 100%|██████████|  260kB /  260kB,  434kB/s  
New Data Upload               : 100%|██████████|  260kB /  260kB,  434kB/s  
                              : 100%|██████████|  260kB /  260kB            
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.41s/it]


> csasr.data.push_to_hub --manifest /kaggle/working/manifests/sentences.jsonl --repo RohanRamesh/hi-en-synth-cs --config sentences --text-only


Creating parquet from Arrow format: 100%|██████████| 16/16 [00:00<00:00, 955.01ba/s]
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

                              :  53%|█████▎    |  530kB / 1.00MB            

Processing Files (0 / 1)      :  53%|█████▎    |  530kB / 1.00MB,   ???B/s  
New Data Upload               :  53%|█████▎    |  530kB / 1.00MB,   ???B/s  

                              :  53%|█████▎    |  530kB / 1.00MB            

Processing Files (1 / 1)      : 100%|██████████| 1.00MB / 1.00MB, 1.19MB/s  
New Data Upload               : 100%|██████████| 1.00MB / 1.00MB, 1.19MB/s  

                              : 100%|██████████| 1.00MB / 1.00MB            

Processing Files (1 / 1)      : 100%|██████████| 1.00MB / 1.00MB,  791kB/s  
New Data Upload               : 100%|██████████| 1.00MB / 1.00MB,  791kB/s  
                              : 100%|██████████| 1.00MB / 1.00MB      


text stage complete -> now run 01b_synthesize_audio.ipynb
